# 05 - 微调后模型评估

加载微调后的模型，运行与微调前相同的评测流程，对比能力提升。

**兼容性**: 本笔记本已适配 Transformers v5 + PEFT v0.18 (含 WeightConverter 修复)

## 5.1 预检与配置

验证adapter、基座模型和pipeline_state可访问。

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import os
import torch
from pathlib import Path
from datetime import datetime
from config import MODELS_DIR, RESULTS_DIR, BASE_MODEL, DATA_DIR

# 路径配置
base_model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
adapter_path = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')
test_data_path = os.path.join(DATA_DIR, 'sample_data.json')

print(f'基座模型路径: {base_model_path}')
print(f'适配器路径: {adapter_path}')

# 预检
errors = []
if not os.path.exists(adapter_path):
    errors.append(f'适配器不存在: {adapter_path}')
if not os.path.exists(base_model_path):
    errors.append(f'基座模型不存在: {base_model_path}')

# pipeline_state 可选 (可能为dict而非对象)
try:
    from utils.pipeline_state import PipelineState
    state = PipelineState()
    baseline = state.get('baseline_results') if hasattr(state, 'get') else None
    if baseline is None:
        print('WARNING: pipeline_state中未找到基线结果，将在后续自动运行快速基线。')
    else:
        print(f'基线结果已加载: {baseline["path"]}')
except Exception as e:
    print(f'WARNING: pipeline_state加载失败 ({e})，跳过基线加载。')
    baseline = None

if errors:
    raise RuntimeError('预检失败:\n' + '\n'.join(errors))
print('预检通过!')

## 5.2 加载微调后的模型 (Adapter)

使用手动加载方式绕过 PEFT v0.18 + Transformers v5 的 WeightConverter bug。

In [ ]:
# 修复 PEFT v0.18 WeightConverter 兼容性 (必须在加载模型前运行)
import peft.utils.transformers_weight_conversion as twc

_original_convert = twc.convert_peft_adapter_state_dict_for_transformers

def _skip_weight_conversion(model, peft_config, adapter_state_dict, adapter_name):
    return adapter_state_dict

twc.convert_peft_adapter_state_dict_for_transformers = _skip_weight_conversion
print('PEFT weight conversion patched')

# 加载模型
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print('加载微调后的模型 (Adapter)...')

tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, adapter_path)

print('Adapter模型加载完成!')
print(f'显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

## 5.3 Layer 2 TRIZ评测 (Adapter)

In [ ]:
from utils.benchmark_utils import run_triz_evaluation

print('运行Layer 2 TRIZ评测 (Adapter)...')

triz_results_after = run_triz_evaluation(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
    test_data_path=test_data_path,
    max_new_tokens=512,
    temperature=0.7,
)

print('\nAdapter TRIZ评测结果:')
print(f'  综合得分: {triz_results_after["overall_score"]:.2%}')
print(f'  原理识别: {triz_results_after["principle_accuracy"]["accuracy"]:.2%}')
print(f'  矛盾解决: {triz_results_after["contradiction_resolution"]["average_score"]:.2%}')
print(f'  案例质量: {triz_results_after["case_quality"]["average_coverage"]:.2%}')
print(f'  ARIZ完整: {triz_results_after["ariz_completeness"]["completeness"]:.2%}')

## 5.4 Layer 3 性能评测 (Adapter)

In [ ]:
from utils.benchmark_utils import run_performance_benchmark

print('运行Layer 3 性能评测 (Adapter)...')

perf_results_after = run_performance_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
)

print('\nAdapter性能评测完成!')
print(f'  吞吐量: {perf_results_after["throughput_tokens_per_sec"]:.1f} tokens/s')
print(f'  P50延迟: {perf_results_after["latency_p50_ms"]:.1f} ms')
print(f'  峰值内存: {perf_results_after["memory_peak_gb"]:.1f} GB')

## 5.x Layer 1 通用能力基准 (Adapter)

In [ ]:
# Layer 1: 通用能力评测 (Adapter) — 为真实 before/after 对比重新运行
from utils.benchmark_utils import run_lm_evaluation
from config import BENCHMARK_CONFIG

tasks = list(BENCHMARK_CONFIG['general_benchmarks'].keys())
print(f'开始通用能力评测 (Layer 1, Adapter): {tasks}')

after_layer1 = run_lm_evaluation(
    model_path='',
    tasks=tasks,
    output_dir=RESULTS_DIR,
    num_fewshot=5,
    batch_size=1,
    model=model,
    tokenizer=tokenizer,
)

print('\nAdapter Layer 1 评测完成!')

## 5.5 实际案例测试 (Adapter)

In [ ]:
from utils.data_utils import format_messages

test_cases = [
    {"name": "矛盾分析", "prompt": "一辆汽车需要既坚固（安全性）又轻便（省油），请用TRIZ分析这个技术矛盾并给出解决方案。"},
    {"name": "原理推荐", "prompt": "如何提高太阳能电池板的能量转换效率？请推荐相关的TRIZ发明原理。"},
    {"name": "ARIZ指导", "prompt": "使用ARIZ算法，分析'如何在不影响打印质量的情况下降低3D打印成本'这个问题。"},
]

for case in test_cases:
    print(f"\n{'='*60}")
    print(f"测试: {case['name']}")
    print(f"{'='*60}")

    prompt = format_messages(
        tokenizer,
        user_content=case['prompt'],
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'assistant' in response:
        response = response.split('assistant')[-1].strip()

    print(response[:800])
    print('... [truncated]')

In [ ]:
# 清理显存，准备加载基座模型
del model
torch.cuda.empty_cache()
print(f'显存已清理: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

## 5.6 加载基座模型 (Base Model)

基座模型无需量化，直接加载FP16。

In [ ]:
from transformers import AutoModelForCausalLM

print('加载基座模型 (无Adapter)...')

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('基座模型加载完成!')
print(f'显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

## 5.7 Layer 2 TRIZ评测 (Base Model)

In [ ]:
from utils.benchmark_utils import run_triz_evaluation

print('运行Layer 2 TRIZ评测 (Base Model)...')

triz_results_before = run_triz_evaluation(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
    test_data_path=test_data_path,
    max_new_tokens=512,
    temperature=0.7,
)

print('\nBase Model TRIZ评测结果:')
print(f'  综合得分: {triz_results_before["overall_score"]:.2%}')
print(f'  原理识别: {triz_results_before["principle_accuracy"]["accuracy"]:.2%}')
print(f'  矛盾解决: {triz_results_before["contradiction_resolution"]["average_score"]:.2%}')
print(f'  案例质量: {triz_results_before["case_quality"]["average_coverage"]:.2%}')
print(f'  ARIZ完整: {triz_results_before["ariz_completeness"]["completeness"]:.2%}')

## 5.8 Layer 3 性能评测 (Base Model)

In [ ]:
from utils.benchmark_utils import run_performance_benchmark

print('运行Layer 3 性能评测 (Base Model)...')

perf_results_before = run_performance_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
)

print('\nBase Model性能评测完成!')
print(f'  吞吐量: {perf_results_before["throughput_tokens_per_sec"]:.1f} tokens/s')
print(f'  P50延迟: {perf_results_before["latency_p50_ms"]:.1f} ms')
print(f'  峰值内存: {perf_results_before["memory_peak_gb"]:.1f} GB')

## 5.9 加载基线结果 (Layer 1)

In [ ]:
# 从pipeline_state加载Layer 1基线结果 (带路径校验)
import json
from config import BENCHMARK_CONFIG
from pathlib import Path

try:
    from utils.pipeline_state import PipelineState
    state = PipelineState()
    baseline = state.get('baseline_results') if hasattr(state, 'get') else None
except Exception as e:
    print(f'WARNING: pipeline_state加载失败 ({e})')
    baseline = None

baseline_layer1 = None
if baseline is not None:
    layer1_path = baseline.get('metadata', {}).get('layer1_path')
    if layer1_path:
        layer1_path = Path(layer1_path).resolve()
        results_dir = Path(RESULTS_DIR).resolve()
        try:
            if layer1_path.is_relative_to(results_dir) and layer1_path.exists():
                with open(layer1_path, 'r', encoding='utf-8') as f:
                    baseline_layer1 = json.load(f)
                print(f'Layer 1基线已加载: {layer1_path}')
            else:
                if not layer1_path.is_relative_to(results_dir):
                    print(f'WARNING: layer1_path {layer1_path} 不在 RESULTS_DIR 下，拒绝加载。')
                else:
                    print(f'WARNING: layer1_path {layer1_path} 不存在。')
        except Exception as e:
            print(f'WARNING: 加载Layer 1基线失败 ({e})')
            baseline_layer1 = None

if baseline_layer1 is None:
    print('WARNING: Baseline Layer 1 file not found — running quick baseline now. Run Notebook 03 for full baseline.')
    # 快速基线只补Layer 1分数；Layer 3性能已由本notebook 5.8节的base model性能评测提供。
    from utils.benchmark_utils import run_lm_evaluation
    quick_task = 'mmlu_pro'
    quick_config = BENCHMARK_CONFIG['general_benchmarks'][quick_task]
    baseline_layer1 = run_lm_evaluation(
        model_path=base_model_path,
        tasks=[quick_task],
        output_dir=RESULTS_DIR,
        num_fewshot=quick_config['num_fewshot'],
        batch_size=1,
    )
    print('快速基线完成 (Layer 1 quick baseline).')
else:
    # 显示基线摘要信息
    print(f'  基线任务: {baseline.get("metadata", {}).get("tasks", [])}')
    print(f'  基线性能吞吐量: {baseline.get("metadata", {}).get("perf_throughput", "N/A")}')

## 5.10 生成对比报告

In [ ]:
from utils.benchmark_utils import aggregate_results
from datetime import datetime

# 构建before/after结果字典
before_results = {
    'layer1_general': baseline_layer1 if 'baseline_layer1' in dir() else {},
    'layer2_triz': triz_results_before if 'triz_results_before' in dir() else {},
    'layer3_performance': perf_results_before if 'perf_results_before' in dir() else {},
}

after_results = {
    'layer1_general': after_layer1 if 'after_layer1' in dir() else {},
    'layer2_triz': triz_results_after if 'triz_results_after' in dir() else {},
    'layer3_performance': perf_results_after if 'perf_results_after' in dir() else {},
}

model_info = {
    'base_model': BASE_MODEL,
    'adapter_path': adapter_path,
    'timestamp': datetime.now().isoformat(),
}

report = aggregate_results(
    general_results=general_results if 'general_results' in dir() else None,
    before_results=before_results,
    after_results=after_results,
    output_dir=RESULTS_DIR,
    model_info=model_info,
)

print('对比报告生成完成!')
print(f'报告保存位置: {RESULTS_DIR}')

## 5.11 结果展示

In [ ]:
from IPython.display import Markdown, display

def display_delta_table(metrics, title):
    lines = [f'### {title}', '', '| 指标 | 微调前 | 微调后 | 变化 | 变化率 |', '|------|--------|--------|------|--------|']
    for key, val in metrics.items():
        if isinstance(val, dict) and 'before' in val and 'after' in val:
            before = val['before']
            after = val['after']
            delta = val.get('delta', 0)
            delta_pct = val.get('delta_pct', 0)
            sign = '+' if delta >= 0 else ''
            lines.append(f"| {key} | {before:.4f} | {after:.4f} | {sign}{delta:.4f} | {sign}{delta_pct:.1f}% |")
    display(Markdown('\n'.join(lines)))

# 显示Layer 1结果
layer1_metrics = report.get('layer1_general', {}).get('metrics', {})
if layer1_metrics:
    display_delta_table(layer1_metrics, 'Layer 1: 通用能力对比')

# 显示Layer 2结果
layer2_metrics = report.get('layer2_triz', {}).get('metrics', {})
if layer2_metrics:
    display_delta_table(layer2_metrics, 'Layer 2: TRIZ能力对比')

# 显示Layer 3结果
layer3_metrics = report.get('layer3_performance', {}).get('metrics', {})
if layer3_metrics:
    display_delta_table(layer3_metrics, 'Layer 3: 性能指标对比')

# 显示综合评分
summary = report.get('summary', {})
print('\n' + '='*60)
print('综合评分')
print('='*60)
for key, value in summary.items():
    print(f'  {key}: {value}')
print('='*60)

In [ ]:
# 清理显存
del model
del tokenizer
torch.cuda.empty_cache()
print(f'显存已清理: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')
print('\n评估完成! 所有结果已保存到 results/ 目录')

---

## 项目完成! 

所有流程已执行完毕。项目文件结构:
```
mongoose_ai/
├── models/
│   ├── Qwen3.6-35B-A3B/          # 基座模型
│   └── meerkat_triz_adapter_v1/  # LoRA适配器 (~100MB)
├── data/
│   ├── raw/                      # 原始数据
│   └── processed/                # ChatML格式数据
├── results/                      # 评测结果
└── checkpoints/                  # 训练检查点
```